## ***Northeast Conference***

***Schools***
1. Central Connecticut State
2. Chicago State
3. Fairleigh Dickinson
4. Le Moyne
5. Long Island
6. Mercyhurst
7. New Haven
8. Saint Francis
9. Stonehill
10. Wagner

In [18]:
### Imports
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

***Scrapers***

In [ ]:
"""
Sidearm Sports <tr> Scraper

Teams:
- Most of Atlantic Sun, Atlantic 10
- Big East: Butler, Georgetown, Villanova
- Big Sky: UC Davis, Eastern Washington, Idaho State, Idaho, Northern Arizona, Portland State, Sacramento State
- Big South: Charleston Southern, Gardner-Webb, High Point, Longwood, Presbyterian, Radford, USC Upstate, Winthrop
- Big West: Cal State Bakersfield, Cal State Fullerton, Hawai'i, UC Irvine, UC Riverside, UC San Diego, UC Santa Barbara
- Big 10: Illinois, Maryland, Rutgers
- CAA: Campbell, Elon, Hampton, Hofstra, Monmouth, North Carolina A&T, Northeastern, Stony Brook, Towson, UNC Wilmington, William & Mary
- Horizon League: Cleveland State, Detroit Mercy, Green Bay, IU Indianapolis, Milwaukee, Northern Kentucky, Oakland, Robert Morris, Wright State, Youngstown State
- Conference USA: FIU, Jacksonville State, Kennesaw State, Louisiana Tech, Missouri State, New Mexico State, Sam Houston, UTEP
- Ivy League: Cornell, Harvard, Pennsylvania, Yale
- Metro Atlantic: Canisius, Fairfield, Iona, Manhattan, Marist, Merrimack, Mount St. Mary's, Niagara, Rider, Sacred Heart, Saint Peter's, Siena
- Mid-American: Northern Illinois, Ohio, Western Michigan
- Mid-Eastern: Whole conference
- Missouri Valley: Belmont, Bradley, Drake, Evansville, UIC, Illinois State, Indiana State, Valparaiso
- Mountain West: Nevada, UNLV, Utah State, Wyoming
- Northeast: Chicago State, Fairleigh Dickinson, Le Moyne, Long Island, Mercyhurst, New Haven, Saint Francis, Stonehill, Wagner
"""
def scrape_sidearm_tr(url, school, conference = "Northeast"):
    
    # Set custom headers to mimic a browser request
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    # Get url, parse
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
    except Exception as e:
        print(f"Error fetching {school}: {e}")
        return pd.DataFrame()

    soup = BeautifulSoup(response.text, "html.parser")

    # Initialize variables
    staff_list = []
    current_department = None

    # Iterate through table rows
    for row in soup.find_all("tr"):
        row_class = [cls.strip() for cls in row.get("class", [])]

        # Department header
        if any("sidearm-staff-category" in c for c in row_class):
            dept_cell = row.find("td", class_="fake-heading")
            if dept_cell:
                current_department = dept_cell.get_text(strip=True)

        # Staff member row
        elif any("sidearm-staff-member" in c for c in row_class):
            name_tag = row.find("a", href=True)
            name = name_tag.get_text(strip=True) if name_tag else None

            title_cell = row.find("td", headers=lambda h: h and "col-staff_title" in h)
            if not title_cell:
                all_tds = row.find_all("td")
                title_cell = all_tds[2] if len(all_tds) > 2 else None
            title = title_cell.get_text(strip=True) if title_cell else None

            if name:
                staff_list.append({
                    "name": name,
                    "title": title or "",
                    "department": current_department or "",
                    "conference": conference,
                    "school": school
                })

    df = pd.DataFrame(staff_list)
    print(f"{school}: Scraped {len(df)} Staff Members")
    return df


In [ ]:
"""
IMPORTANT: requires Selenium

Each department is under a <section aria-label="..." class="staff-directory-group"> tag.
We can loop through each section. The department name is stored in a <h2> tag. 
Then, the staff members and titles are in a <tbody> tag in the current section.
Each staff member card is a <tr> tag, and we access the name through a <td data-title="Name">, and scrape the <a> tag.
Each title is in a <td data-title="Title"> tag, and scrape a <div> tag for the text.

Schools:
- Northeast: Central Connecticut State
"""

def scrape_section_h2_a_div_selenium(url, school, conference="Northeast"):
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

    try:
        driver.get(url)

        # Wait until at least one staff section loads
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CLASS_NAME, "staff-directory-group"))
        )

        html = driver.execute_script("return document.body.innerHTML")
        soup = BeautifulSoup(html, "html.parser")

    finally:
        driver.quit()

    staff_list = []

    # Loop through each department section
    for section in soup.find_all("section", class_="staff-directory-group"):
        # Department name
        h2 = section.find("h2")
        department = h2.get_text(strip=True) if h2 else section.get("aria-label", "")

        # Staff rows
        tbody = section.find("tbody")
        if not tbody:
            continue

        for row in tbody.find_all("tr"):
            # Name
            name_cell = row.find("td", {"data-title": "Name"})
            name_tag = name_cell.find("a") if name_cell else None
            name = name_tag.get_text(strip=True) if name_tag else None

            # Title
            title_cell = row.find("td", {"data-title": "Title"})
            title_div = title_cell.find("div") if title_cell else None
            title = title_div.get_text(strip=True) if title_div else ""

            if name:
                staff_list.append({
                    "name": name,
                    "title": title,
                    "department": department,
                    "school": school,
                    "conference": conference
                })

    df = pd.DataFrame(staff_list)
    print(f"{school} scraped {len(df)} staff members.")
    return df

***Individual School Checks***

In [ ]:
### Central Connecticut State
school_df = scrape_section_h2_a_div_selenium(url = 'https://ccsubluedevils.com/athletics/directory/index', school = 'Central Connecticut State')
school_df['department'].value_counts()

In [ ]:
### Chicago State
school_df = scrape_sidearm_tr(url = 'https://www.gocsucougars.com/staff-directory', school = 'Chicago State')
school_df['department'].value_counts()

In [ ]:
### Fairleigh Dickinson
school_df = scrape_sidearm_tr(url = 'https://fduknights.com/staff-directory', school = 'Fairleigh Dickinson')
school_df['department'].value_counts()

In [ ]:
### Le Moyne
school_df = scrape_sidearm_tr(url = 'https://lemoynedolphins.com/staff-directory', school = 'Le Moyne')
school_df['department'].value_counts()

In [ ]:
### Long Island
school_df = scrape_sidearm_tr(url = 'https://www.liuathletics.com/staff-directory', school = 'Long Island')
school_df['department'].value_counts()

In [ ]:
### Mercyhurst
school_df = scrape_(url = '', school = 'Mercyhurst')
school_df['department'].value_counts()

In [ ]:
### New Haven
school_df = scrape_(url = '', school = 'New Haven')
school_df['department'].value_counts()

In [ ]:
### Saint Francis
school_df = scrape_(url = '', school = 'Saint Francis')
school_df['department'].value_counts()

In [ ]:
### Stonehill
school_df = scrape_(url = '', school = 'Stonehill')
school_df['department'].value_counts()

In [ ]:
### Wagner
school_df = scrape_(url = '', school = 'Wagner')
school_df['department'].value_counts()

***Scrape Northeast***

In [20]:
# Whole conference scraper
def scrape_northeast():
    schools = [
        {'school': 'Central Connecticut State', 'url': 'https://ccsubluedevils.com/athletics/directory/index', 'scraper': 'section_h2_a_div', 'conference': 'Northeast'},
        {'school': 'Chicago State', 'url': 'https://www.gocsucougars.com/staff-directory', 'scraper': 'sidearm', 'conference': 'Northeast'},
        {'school': 'Fairleigh Dickinson', 'url': 'https://fduknights.com/staff-directory', 'scraper': 'sidearm', 'conference': 'Northeast'},
        {'school': 'Le Moyne', 'url': 'https://lemoynedolphins.com/staff-directory', 'scraper': 'sidearm', 'conference': 'Northeast'},
        {'school': 'Long Island', 'url': 'https://www.liuathletics.com/staff-directory', 'scraper': 'sidearm', 'conference': 'Northeast'},
        {'school': 'Mercyhurst', 'url': 'https://hurstathletics.com/staff-directory', 'scraper': 'sidearm', 'conference': 'Northeast'},
        {'school': 'New Haven', 'url': 'https://newhavenchargers.com/staff-directory', 'scraper': 'sidearm', 'conference': 'Northeast'},
        {'school': 'Saint Francis', 'url': 'https://sfuathletics.com/staff-directory', 'scraper': 'sidearm', 'conference': 'Northeast'},
        {'school': 'Stonehill', 'url': 'https://stonehillskyhawks.com/staff-directory', 'scraper': 'sidearm', 'conference': 'Northeast'},
        {'school': 'Wagner', 'url': 'https://wagnerathletics.com/staff-directory', 'scraper': 'sidearm', 'conference': 'Northeast'},
    ]    
    
    # Scrape all schools
    # Add other scrapers if necessary here
    all_staff = []
    for s in schools:
        if s['scraper'] == 'sidearm':
            df = scrape_sidearm_tr(url = s['url'], school = s['school'], conference = s['conference'])
        elif s['scraper'] == 'section_h2_a_div':
            df = scrape_section_h2_a_div_selenium(url = s['url'], school = s['school'], conference = s['conference'])
            
        all_staff.append(df)
        
    ### Combine all dataframes, return it
    combined_df = pd.concat(all_staff, ignore_index=True)
    return combined_df

In [21]:
ne_df = scrape_northeast()
ne_df.to_csv("northeast_staff_directory.csv", index=False)

Central Connecticut State scraped 100 staff members.
Chicago State: Scraped 52 Staff Members
Fairleigh Dickinson: Scraped 122 Staff Members
Le Moyne: Scraped 82 Staff Members
Long Island: Scraped 170 Staff Members
Mercyhurst: Scraped 138 Staff Members
New Haven: Scraped 122 Staff Members
Saint Francis: Scraped 108 Staff Members
Stonehill: Scraped 123 Staff Members
Wagner: Scraped 125 Staff Members
